In [21]:
import torch
from torch.utils.data import Dataset
from torchvision.datasets import FashionMNIST

from torchvision import transforms, datasets



class SimCLRFMnist(Dataset):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(28, scale=(0.7, 0.7)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])
    def __init__(self, dataset = FashionMNIST(root="./data", train=True, download=True)):
        self.dataset = dataset
        
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        xi = self.train_transform(img)
        xj = self.train_transform(img)
        return xi, xj

    def __len__(self):
        return len(self.dataset)
    
class ClassifierTrain(Dataset):
    train_transform = transforms.Compose([
        transforms.Resize(28),
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])
    def __init__(self, dataset = FashionMNIST(root="./data", train=True, download=True)):
        self.dataset = dataset
    
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = self.train_transform(img)
        return img, label
    
    def __len__(self):
        return len(self.dataset)

In [22]:
from torch.utils.data import DataLoader

train_dataset = SimCLRFMnist()
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

In [23]:
import copy
import torch
import torch.nn as nn
import timm
from timm.models.vision_transformer import VisionTransformer


class SimCLR(nn.Module):
    def __init__(
        self,
        img_size=28,
        patch_size=4,
        embed_dim=96,
        depth=3,
        num_heads=4,
        in_chans=1,
        hidden_dim=128,
        projection_dim=64
    ):
        super().__init__()
        
        self.encoder = VisionTransformer(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            num_classes=0,
            in_chans=in_chans,
        )
        self.projector = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, projection_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        return h, nn.functional.normalize(self.projector(h), dim=1)
    
class Classifier(nn.Module):
    def __init__(self, encoder_backbone, device, num_classes, from_scratch = False):
        super().__init__()
        self.backbone = copy.deepcopy(encoder_backbone).to(device)
        if from_scratch:
            for m in self.backbone.modules():
                if hasattr(m, 'reset_parameters'):
                    print("reseting parameters")
                    m.reset_parameters()
        else:
            for param in self.backbone.parameters():
                param.requires_grad = False
        self.classifier = nn.Linear(encoder_backbone.num_features, num_classes).to(device)
        
    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)
        
        


In [24]:
from torch.utils.tensorboard import SummaryWriter

class BaseLogger:
    def __init__(self):
        pass
    
    def log(self, epoch, **kwargs):
        print(f"Epoch: {epoch}")
        for key, val in kwargs.items():
            print(f"{key}: {val}")
            
class TensorBoardLogger(BaseLogger):
    def __init__(self):
        self.writer = SummaryWriter()
        
    def log(self,epoch, **kwargs):
        print(f"Epoch: {epoch}")
        for key, val in kwargs.items():
            print(f"{key}: {val}")
            self.writer.add_scalar(key,val,epoch - 1)
        self.writer.flush()

In [25]:




from tqdm import tqdm
from torch import optim
from torch import nn
from torch.nn import functional as F

class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, zi, zj):
        batch_size = zi.size(0)
        z = torch.cat([zi, zj], dim=0)
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
        sim = sim / self.temperature

        sim_i_j = torch.diag(sim, batch_size)
        sim_j_i = torch.diag(sim, -batch_size)
        positives = torch.cat([sim_i_j, sim_j_i], dim=0)
        negatives = sim[~mask].view(2 * batch_size, -1)

        labels = torch.zeros(2 * batch_size, dtype=torch.long, device=z.device)
        logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
        loss = F.cross_entropy(logits, labels)
        return loss


class SimCLRTrainer:
    def __init__(
        self,
        model,
        criterion=NTXentLoss(temperature=0.5),
        logger : BaseLogger=BaseLogger(),
        device = "cpu"
    ):
        self.device = device
        self.model = model.to(device)
        self.criterion = criterion
        self.logger = logger
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.0005)
        
    def train(self,train_loader, num_epochs):
        for epoch in range(num_epochs):
            self.model.train()
            total_loss = 0
            for xi, xj in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
                xi, xj = xi.to(self.device), xj.to(self.device)
                _, zi = self.model(xi)
                _, zj = self.model(xj)

                loss = self.criterion(zi, zj)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            self.logger.log(epoch=epoch + 1, loss=total_loss/len(train_loader))

In [27]:
model = SimCLR(img_size=28, depth=3, embed_dim=96, patch_size=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
simCLR_trainer = SimCLRTrainer(
    model=model,
    criterion=NTXentLoss(temperature=0.3),
    logger=TensorBoardLogger(), # use BaseLogger if tensorboard not installed
    device=device
)
num_epochs = 10
simCLR_trainer.train(train_loader, num_epochs)

Epoch 1:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 1: 100%|██████████| 469/469 [02:08<00:00,  3.64it/s]


Epoch: 1
loss: 3.258201457289999


Epoch 2: 100%|██████████| 469/469 [02:10<00:00,  3.59it/s]


Epoch: 2
loss: 2.955454770944266


...

Epoch: 8
loss: 2.7430006090257724


Epoch 9: 100%|██████████| 469/469 [02:09<00:00,  3.63it/s]


Epoch: 9
loss: 2.726388032502457


Epoch 10: 100%|██████████| 469/469 [02:09<00:00,  3.62it/s]

Epoch: 10
loss: 2.713232156310254


In [28]:
dataset = ClassifierTrain()

train_loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)

In [29]:
classifier = Classifier(model.encoder, device = device, num_classes=10)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
for epoch in range(10):
    classifier.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = classifier(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

Epoch 1: loss=1.2440
Epoch 2: loss=0.9104
Epoch 3: loss=0.8535
Epoch 4: loss=0.8272
Epoch 5: loss=0.8107
Epoch 6: loss=0.7995
Epoch 7: loss=0.7904
Epoch 8: loss=0.7840
Epoch 9: loss=0.7781
Epoch 10: loss=0.7730


In [31]:

classifier.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = classifier(x).argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 71.38%
